# Demo A: Why Serving an LLM Is Hard — Measured, Start to Finish

**ODSC 2026** | Serving LLMs Efficiently at Scale | [Harshul Jain](https://github.com/harshuljain13)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/odsc-2026/demos/odsc_2026_demo_a_model_memory_prefill_decode.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/odsc-2026/demos/odsc_2026_demo_a_model_memory_prefill_decode.ipynb)

## One chain of reasoning, six measurements

This notebook is not a tour of six topics. It is **one argument**, where each section
exists only because the previous one raised a question:

```
  §1  We download a model.  It is N GB of weights.
        └─ so what else needs memory, and how much is left?
  §2  Weights are a FIXED cost. The KV cache is the PER-USER one.
        └─ fine — but what does the user actually experience?
  §3  We serve two prompts. The long one waits far longer for its first token.
        └─ where inside that wait did the time go?
  §4  Open the box: 2 prompts x 2 cache settings = 4 runs, prefill and decode
      timed separately. The cache does NOTHING for prefill and EVERYTHING
      for decode. With it, decode is flat; without it, decode grows per token.
        └─ decode is flat but slow. What is it waiting on?
  §5  Memory bandwidth. Each decoded token re-reads every weight.
      We measure this GPU's bandwidth and compute the floor.
        └─ so what does that cache actually weigh?
  §6  Weigh it. It matches the §2 prediction — so you can size this from
      config.json alone.
        └─ floor = (weights + KV) / bandwidth. Shrink both. That is Demo B.
```

**Ground rule: every number is measured here, on this machine.** The one value we
*predict* from `config.json` (KV bytes per token, §2) is labelled a prediction and then
weighed against the real cache in §6.

**Hardware:** a CUDA GPU with ≥ 18 GB runs the 7B path. Anything smaller, or CPU-only,
falls back to a 1.1B model and smaller knobs — same shape of result, different absolute
numbers.

## 0. Setup

Three cells of scaffolding — hardware, knobs, helpers — then the story starts and does
not stop for plumbing again.

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers', 'torch', 'matplotlib', 'pandas', 'accelerate'])

import gc
import time

import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device('cuda' if HAS_CUDA else 'cpu')

if HAS_CUDA:
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Device : {GPU_NAME} ({VRAM_GB:.1f} GB VRAM)')
else:
    GPU_NAME, VRAM_GB = 'CPU', 0.0
    print('Device : CPU (no CUDA). Slower, but the asymmetry still shows.')
print(f'torch  : {torch.__version__}')

In [ ]:
# --- Knobs, scaled to the hardware so this finishes in minutes either way ---
BIG, SMALL = 'mistralai/Mistral-7B-v0.1', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
USE_BIG = HAS_CUDA and VRAM_GB >= 18

MODEL_ID = BIG if USE_BIG else SMALL
DTYPE = torch.float16 if HAS_CUDA else torch.float32
SHORT_PROMPT = 'Explain quantum computing in simple terms:'
FILLER = 'The quick brown fox jumps over the lazy dog. '
LONG_TOKENS = 4096 if USE_BIG else (1024 if HAS_CUDA else 256)
DECODE_STEPS = 32 if HAS_CUDA else 8
NO_CACHE_STEPS = 12 if HAS_CUDA else 4
SWEEP = [128, 512, 1024, 2048, 4096] if USE_BIG else (
    [128, 256, 512, 1024] if HAS_CUDA else [64, 128, 256])

print(f'Model       : {MODEL_ID}')
print(f'Precision   : {str(DTYPE).replace("torch.", "")}')
print(f'Long prompt : {LONG_TOKENS} tokens | decode steps: {DECODE_STEPS}')

In [ ]:
# --- Plot helpers ---------------------------------------------------------
# Every chart in this notebook is one call into these. Keeps the story cells
# readable instead of burying the point under axis bookkeeping.
PALETTE = ['#dbeafe', '#dcfce7', '#fef3c7', '#ffedd5', '#ffe4e6', '#f3e8ff']
COOL, WARM = '#2563eb', '#991b1b'


def panels(n=1, width=6.5, height=4.5):
    fig, axes = plt.subplots(1, n, figsize=(width * n, height))
    return (axes,) if n == 1 else axes


def _finish(ax, title, xlabel=None, ylabel=None, legend=False):
    ax.set_title(title, fontsize=12, fontweight='bold')
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=11)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=11)
    if legend:
        ax.legend(fontsize=9)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)


def bars(ax, labels, values, title, ylabel, fmt='{:.0f}'):
    values = list(values)
    ax.bar(range(len(values)), values, color=PALETTE[:len(values)],
           edgecolor='#000', linewidth=1.2)
    ax.set_xticks(range(len(values)))
    ax.set_xticklabels(labels)
    for i, v in enumerate(values):
        ax.text(i, v, fmt.format(v), ha='center', va='bottom', fontsize=10)
    ax.set_ylim(0, max(values) * 1.25)
    _finish(ax, title, ylabel=ylabel)


def hbars(ax, labels, values, title, xlabel, notes=None, fmt='{:.2f}'):
    values = list(values)
    ax.barh(range(len(values)), values, color=PALETTE[:len(values)],
            edgecolor='#000', linewidth=1.2)
    ax.set_yticks(range(len(values)))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    for i, v in enumerate(values):
        label = fmt.format(v) + (f'  ({notes[i]})' if notes else '')
        ax.text(v + max(values) * 0.02, i, label, va='center', fontsize=9)
    ax.set_xlim(0, max(values) * 1.4)
    _finish(ax, title, xlabel=xlabel)


def stacked(ax, labels, parts, title, ylabel, fmt=None):
    # parts: [(series_name, [values aligned to labels]), ...]
    bottom = [0.0] * len(labels)
    for i, (name, vals) in enumerate(parts):
        ax.bar(labels, vals, bottom=bottom, color=PALETTE[i % len(PALETTE)],
               edgecolor='#000', linewidth=1.2, label=name)
        bottom = [b + v for b, v in zip(bottom, vals)]
    if fmt:
        for i, total in enumerate(bottom):
            ax.text(i, total, fmt.format(total), ha='center', va='bottom', fontsize=9)
    _finish(ax, title, ylabel=ylabel, legend=True)


def grouped(ax, groups, series, title, ylabel, fmt='{:.1f}'):
    # series: [(series_name, [values aligned to groups]), ...]
    width = 0.8 / len(series)
    for si, (name, vals) in enumerate(series):
        xs = [g + si * width - 0.4 + width / 2 for g in range(len(groups))]
        ax.bar(xs, vals, width, color=PALETTE[si % len(PALETTE)],
               edgecolor='#000', linewidth=1.1, label=name)
        for x, v in zip(xs, vals):
            ax.text(x, v, fmt.format(v), ha='center', va='bottom', fontsize=9)
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(groups)
    _finish(ax, title, ylabel=ylabel, legend=True)


def curves(ax, series, title, xlabel, ylabel, logy=False, zero_floor=True):
    # series: [(name, xs, ys, color_or_None), ...]
    for i, (name, xs, ys, color) in enumerate(series):
        ax.plot(xs, ys, marker='o', markersize=3.5, linewidth=1.8,
                color=color or [COOL, WARM][i % 2], label=name)
    if logy:
        ax.set_yscale('log')
    elif zero_floor:
        ax.set_ylim(bottom=0)
    _finish(ax, title, xlabel=xlabel, ylabel=ylabel, legend=len(series) > 1)


print('Plot helpers ready: bars, hbars, stacked, grouped, curves, panels.')


## §1 · We Download a Model. How Big Is It, Really?

Start where every deployment starts: a Hugging Face model id. Load it, and get the two
numbers the rest of the notebook depends on — **parameter count** and **bytes on the
device**.

In [ ]:
config = AutoConfig.from_pretrained(MODEL_ID)
N_LAYERS = config.num_hidden_layers
N_HEADS = config.num_attention_heads
N_KV_HEADS = getattr(config, 'num_key_value_heads', None) or N_HEADS
HEAD_DIM = getattr(config, 'head_dim', None) or (config.hidden_size // N_HEADS)

print(f'Layers      : {N_LAYERS}')
print(f'Hidden dim  : {config.hidden_size}')
print(f'Q / KV heads: {N_HEADS} / {N_KV_HEADS}' +
      ('   <- GQA: KV side is narrower than Q' if N_KV_HEADS < N_HEADS else '   (no GQA)'))
print(f'Head dim    : {HEAD_DIM}')

In [ ]:
print(f'Loading {MODEL_ID} ... (first run downloads several GB)')

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map='auto' if HAS_CUDA else None).eval()
if not HAS_CUDA:
    model.to(DEVICE)
LOAD_SEC = time.perf_counter() - t0

N_PARAMS = sum(p.numel() for p in model.parameters())
WEIGHT_BYTES = sum(p.numel() * p.element_size() for p in model.parameters())

print(f'\nLoad time   : {LOAD_SEC:.1f} s')
print(f'Parameters  : {N_PARAMS/1e9:.2f} B at {WEIGHT_BYTES/N_PARAMS:.1f} bytes each')
print(f'WEIGHTS     : {WEIGHT_BYTES/1e9:.2f} GB  <-- carry this number to §5')

In [ ]:
# --- Measurement helpers --------------------------------------------------
def sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


def timed(fn):
    # Wall time of fn(), with the device synchronized on both sides so we are
    # not timing an async launch queue.
    sync()
    t0 = time.perf_counter()
    result = fn()
    sync()
    return result, time.perf_counter() - t0


def prompt_of(n_tokens):
    # Repeat filler text, truncate to an exact token budget.
    enc = tokenizer(FILLER * max(1, n_tokens // 8), return_tensors='pt',
                    truncation=True, max_length=n_tokens)
    return enc['input_ids'].to(DEVICE)


# transformers has shipped several KV cache representations (legacy tuples,
# key_cache/value_cache lists, newer .layers objects). Normalize to (k, v) pairs
# so the measurements below survive a version bump.
def cache_layers(past):
    if past is None:
        return []
    if hasattr(past, 'to_legacy_cache'):
        try:
            legacy = past.to_legacy_cache()
            if legacy:
                return [(lay[0], lay[1]) for lay in legacy]
        except Exception:
            pass
    if hasattr(past, 'key_cache') and hasattr(past, 'value_cache'):
        return list(zip(past.key_cache, past.value_cache))
    if hasattr(past, 'layers'):
        try:
            return [(lay.keys, lay.values) for lay in past.layers]
        except Exception:
            pass
    if isinstance(past, (list, tuple)):
        return [(lay[0], lay[1]) for lay in past if lay is not None and len(lay) >= 2]
    return []


def cache_bytes(past):
    return sum(k.numel() * k.element_size() + v.numel() * v.element_size()
               for k, v in cache_layers(past))


def cache_tokens(past):
    if past is not None and hasattr(past, 'get_seq_length'):
        try:
            return int(past.get_seq_length())
        except Exception:
            pass
    lays = cache_layers(past)
    return int(lays[0][0].shape[-2]) if lays else 0


def next_token(out):
    return out.logits[:, -1, :].argmax(dim=-1, keepdim=True)


print('Measurement helpers ready: timed, prompt_of, cache_bytes, cache_tokens.')


**Where we are:** we know the fixed cost. But a GPU holding only weights serves
nobody. What else needs room, and how much room is left?

## §2 · Fixed Cost vs Per-User Cost

Weights are paid once and shared by everyone. The **KV cache** is different: every user
needs their own, and it grows with every token they send or receive. That difference —
fixed vs per-user — is the single most important fact in LLM serving economics.

First, where the fixed cost actually sits. We bucket every parameter tensor by name,
which is an exact accounting of the loaded model rather than a guess from the config.

In [ ]:
def weight_breakdown(model):
    # Priority matters: 'lm_head' before the generic attention/MLP patterns.
    RULES = [('LM head', ('lm_head',)),
             ('Embeddings', ('embed_tokens', 'wte', 'word_embeddings', 'embeddings')),
             ('Norms', ('norm', 'ln_')),
             ('Attention (Q,K,V,O)', ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'attn')),
             ('MLP / FFN', ('mlp', 'gate_proj', 'up_proj', 'down_proj', 'fc1', 'fc2', 'ffn'))]

    totals = {}
    for name, p in model.named_parameters():
        low = name.lower()
        bucket = next((b for b, keys in RULES if any(k in low for k in keys)), 'Other')
        totals[bucket] = totals.get(bucket, 0) + p.numel() * p.element_size()
    return dict(sorted(totals.items(), key=lambda kv: -kv[1]))


WEIGHTS_BY_PART = weight_breakdown(model)

weights_table = pd.DataFrame([
    {'Component': part, 'Memory (GB)': round(b / 1e9, 3),
     'Share': f'{b/WEIGHT_BYTES*100:.1f}%'}
    for part, b in WEIGHTS_BY_PART.items()])
weights_table

In [ ]:
if HAS_CUDA:
    sync()
    FREE_B, TOTAL_B = torch.cuda.mem_get_info()
    print(f'Sum of weight tensors : {WEIGHT_BYTES/1e9:.2f} GB')
    print(f'torch allocated       : {torch.cuda.memory_allocated()/1e9:.2f} GB')
    print(f'Overhead vs weights   : '
          f'{torch.cuda.memory_allocated()/1e9 - WEIGHT_BYTES/1e9:.2f} GB')
    print(f'\nFree VRAM             : {FREE_B/1e9:.2f} GB of {TOTAL_B/1e9:.2f} GB')
    print('Everything per-user — KV cache, activations, fragmentation — lives in there.')
else:
    FREE_B = TOTAL_B = 0
    print(f'Weights in host RAM: {WEIGHT_BYTES/1e9:.2f} GB (CPU run, no VRAM budget)')

In [ ]:
fig = plt.figure(figsize=(13, 4.5))
ax_parts = fig.add_subplot(1, 2, 1)
ax_budget = fig.add_subplot(1, 2, 2)

hbars(ax_parts, list(WEIGHTS_BY_PART), [b / 1e9 for b in WEIGHTS_BY_PART.values()],
      f'The {N_PARAMS/1e9:.1f}B weights, by component', 'GB',
      notes=list(weights_table['Share']), fmt='{:.2f} GB')

if HAS_CUDA:
    stacked(ax_budget, ['GPU'], [('Weights + overhead', [(TOTAL_B - FREE_B) / 1e9]),
                                 ('Free for users', [FREE_B / 1e9])],
            'Budget after loading\n(zero users served)', 'VRAM (GB)', fmt='{:.0f} GB')
else:
    ax_budget.axis('off')
    ax_budget.text(0.5, 0.5, 'CPU run —\nno VRAM budget', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

### The per-user cost, predicted

Now the variable side. One cached token costs:

$$\text{bytes per token} = 2 \times L \times n_{kv} \times d_h \times \text{bytes per element}$$

Leading 2 = one K vector plus one V vector. Note `n_kv`, **not** `n_heads` — that
substitution is exactly what GQA buys, and it is Demo B's opening move.

This is the only number in the notebook we do not measure first. Treat it as a
hypothesis; §6 weighs the real cache and checks it.

In [ ]:
ELEM_BYTES = torch.finfo(DTYPE).bits // 8
KV_BYTES_PER_TOKEN_PREDICTED = 2 * N_LAYERS * N_KV_HEADS * HEAD_DIM * ELEM_BYTES

print('PREDICTION (checked in §6)')
print(f'  2 x {N_LAYERS} layers x {N_KV_HEADS} kv-heads x {HEAD_DIM} dim x {ELEM_BYTES} B'
      f' = {KV_BYTES_PER_TOKEN_PREDICTED/1024:.0f} KB per token')
print(f'  one user at {LONG_TOKENS} tokens -> '
      f'{KV_BYTES_PER_TOKEN_PREDICTED*LONG_TOKENS/1e9:.2f} GB')
if HAS_CUDA:
    seats = int(FREE_B / (KV_BYTES_PER_TOKEN_PREDICTED * LONG_TOKENS))
    print(f'  free VRAM therefore holds roughly {seats} such users')

**Where we are:** we can account for every byte, fixed and per-user. None of that is
what a user notices. What they notice is waiting — so let's make them wait.

## §3 · Two Prompts, One Model, Very Different Waits

Same model, same number of output tokens. The only difference is input length. We
measure **time to first token** — the dead air before anything appears — and total time.

In [ ]:
short_ids = tokenizer(SHORT_PROMPT, return_tensors='pt')['input_ids'].to(DEVICE)
long_ids = prompt_of(LONG_TOKENS)
PROMPTS = {'short': short_ids, 'long': long_ids}

# Warm up on the LONGEST input: the first call pays for kernel compilation and peak
# allocation, and timing that would be timing the wrong thing.
_, warm_sec = timed(lambda: model(input_ids=long_ids, use_cache=True))
if HAS_CUDA:
    torch.cuda.empty_cache()

print(f'short : {short_ids.shape[1]:>5} tokens')
print(f'long  : {long_ids.shape[1]:>5} tokens')
print(f'(warmup pass took {warm_sec*1000:.0f} ms and is discarded)')

In [ ]:
def generate_timed(ids, n_new):
    gen = lambda: model.generate(input_ids=ids, max_new_tokens=n_new, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
    out, sec = timed(gen)
    return out, sec


latency = []
for label, ids in PROMPTS.items():
    _, ttft_sec = generate_timed(ids, 1)                 # stop after token 1
    out, total_sec = generate_timed(ids, DECODE_STEPS)   # run to completion
    latency.append({'Prompt': label, 'Input tokens': ids.shape[1],
                    'Output tokens': out.shape[1] - ids.shape[1],
                    'TTFT (ms)': round(ttft_sec * 1000, 1),
                    'Total (ms)': round(total_sec * 1000, 1),
                    'TTFT share': f'{ttft_sec/total_sec*100:.0f}%'})
    print(f'{label:>5}: TTFT {ttft_sec*1000:7.1f} ms | total {total_sec*1000:8.1f} ms')

latency_table = pd.DataFrame(latency)
latency_table

In [ ]:
labels = [f"{r['Prompt']}\n({r['Input tokens']} tok)" for _, r in latency_table.iterrows()]
ttft = list(latency_table['TTFT (ms)'])
after = [t - f for t, f in zip(latency_table['Total (ms)'], ttft)]

ax1, ax2 = panels(2)
bars(ax1, labels, ttft, 'The wait before anything appears', 'TTFT (ms)', '{:.0f} ms')
stacked(ax2, labels, [('Waiting for token 1', ttft),
                      (f'Streaming the next {DECODE_STEPS-1}', after)],
        'Same output budget, different shape', 'Wall time (ms)', fmt='{:.0f} ms')
plt.tight_layout()
plt.show()

print(f"Long prompt waits {ttft[1]/max(ttft[0], 1e-9):.1f}x longer for its first token,")
print(f"and TTFT goes from {latency_table['TTFT share'][0]} of its total to "
      f"{latency_table['TTFT share'][1]}.")

**Where we are:** a measured, user-visible problem. `generate()` reports one number,
though, and one number cannot tell us *which part* got slower. So we stop using it.

## §4 · Open the Box: Four Runs, Two Phases, One Table

We drive the model by hand — the only way to time the phases separately:

| Phase | The call | Tokens per pass | Produces |
|-------|----------|-----------------|----------|
| **Prefill** | `model(prompt_ids)` — one pass | all of them, in parallel | the first token, i.e. **TTFT** |
| **Decode** | one pass per token | exactly 1 (cached) or **the whole sequence again** (uncached) | each later token |

And we run the full grid, because the interesting comparison is two-dimensional:

|  | **with KV cache** | **without KV cache** |
|--|-------------------|----------------------|
| **short prompt** | run 1 | run 2 |
| **long prompt** | run 3 | run 4 |

Four runs, each reporting prefill time *and* per-token decode time. Same step count
everywhere so the means are comparable.

In [ ]:
CMP_STEPS = NO_CACHE_STEPS   # same budget for all four runs — comparable means


@torch.no_grad()
def run_phases(prompt_ids, steps, use_cache):
    # PREFILL — identical work either way; only whether we KEEP the cache differs.
    out, prefill_sec = timed(lambda: model(input_ids=prompt_ids, use_cache=use_cache))
    tok = next_token(out)
    past = out.past_key_values if use_cache else None
    seq = prompt_ids

    # DECODE — cached: feed 1 token + past. Uncached: re-forward the whole sequence.
    step_secs = []
    for _ in range(steps):
        if use_cache:
            out, sec = timed(lambda: model(input_ids=tok, past_key_values=past,
                                           use_cache=True))
            past = out.past_key_values
        else:
            seq = torch.cat([seq, tok], dim=1)
            out, sec = timed(lambda: model(input_ids=seq, use_cache=False))
        step_secs.append(sec)
        tok = next_token(out)

    n_in = int(prompt_ids.shape[1])
    return {'in_tokens': n_in, 'cache': use_cache,
            'prefill_sec': prefill_sec, 'prefill_tok_s': n_in / prefill_sec,
            'step_secs': step_secs,
            'decode_sec': sum(step_secs),
            'itl_ms': sum(step_secs) / len(step_secs) * 1000,
            'decode_tok_s': len(step_secs) / sum(step_secs),
            'kv_bytes': cache_bytes(past), 'kv_tokens': cache_tokens(past)}


grid = {}
for label, ids in PROMPTS.items():
    for use_cache in (True, False):
        print(f'running {label:>5} prompt, cache={"on " if use_cache else "off"} ...')
        grid[(label, use_cache)] = run_phases(ids, CMP_STEPS, use_cache)
        if HAS_CUDA:
            torch.cuda.empty_cache()

runs = {k: grid[(k, True)] for k in PROMPTS}    # cached runs, reused in §5 and §6
print('done.')

In [ ]:
grid_table = pd.DataFrame([{
    'Prompt': label,
    'Input tokens': r['in_tokens'],
    'KV cache': 'on' if use_cache else 'off',
    'Prefill (ms)': round(r['prefill_sec'] * 1000, 1),
    'Prefill (tok/s)': round(r['prefill_tok_s']),
    'Decode (ms/token)': round(r['itl_ms'], 2),
    'Decode (tok/s)': round(r['decode_tok_s'], 1),
    'First step (ms)': round(r['step_secs'][0] * 1000, 2),
    'Last step (ms)': round(r['step_secs'][-1] * 1000, 2),
} for (label, use_cache), r in grid.items()])
grid_table

In [ ]:
# Three views of the same four runs.
fig = plt.figure(figsize=(16, 4.5))
ax1, ax2, ax3 = (fig.add_subplot(1, 3, i) for i in (1, 2, 3))
names = list(PROMPTS)

grouped(ax1, names,
        [('cache on', [grid[(n, True)]['prefill_sec'] * 1000 for n in names]),
         ('cache off', [grid[(n, False)]['prefill_sec'] * 1000 for n in names])],
        'PREFILL: the cache changes nothing', 'ms', fmt='{:.0f}')

grouped(ax2, names,
        [('cache on', [grid[(n, True)]['itl_ms'] for n in names]),
         ('cache off', [grid[(n, False)]['itl_ms'] for n in names])],
        'DECODE: the cache changes everything', 'ms per token', fmt='{:.1f}')
ax2.set_yscale('log')

curves(ax3, [(f'{n}, cache {"on" if c else "off"}',
              range(1, CMP_STEPS + 1),
              [s * 1000 for s in grid[(n, c)]['step_secs']],
              {('short', True): '#93c5fd', ('short', False): COOL,
               ('long', True): '#fca5a5', ('long', False): WARM}[(n, c)])
             for n in names for c in (True, False)],
       'Per-token cost, step by step', 'Decode step', 'ms per token', logy=True)
plt.tight_layout()
plt.show()

### This is the hinge of the whole notebook — read all three panels

**Panel 1 — prefill ignores the cache.** Of course it does: prefill processes the prompt
in one parallel pass either way. The only difference is whether we *keep* the K and V it
produced. So the cache is not a prefill optimization at all, and the long prompt is
slower purely because there is more prompt.

**Panel 2 — decode lives or dies by it.** Note the log scale. With the cache, per-token
cost is nearly the same for both prompts. Without it, cost explodes — *and the long
prompt is hit far harder*, because every single step re-forwards its entire context.

**Panel 3 — the shapes say why.** Cached lines are flat. Uncached lines climb with every
step, since the sequence being re-forwarded grows by one token each time. Compare the
`First step` and `Last step` columns in the table above: flat vs climbing.

Put together: **the cache converts O(n) work per token into O(1).** That is what makes
decode flat.

But flat is not the same as cheap. Even cached, decode costs far more per token than
prefill does — look at the two `tok/s` columns. Something expensive happens on every
step regardless of context, and §5 goes after it.

First, confirm prefill's scaling is not a fluke of two hand-picked prompts.

In [ ]:
sweep = []
for n in SWEEP:
    ids = prompt_of(n)
    r = run_phases(ids, max(8, DECODE_STEPS // 4), True)
    sweep.append({'Input tokens': r['in_tokens'],
                  'Prefill (ms)': round(r['prefill_sec'] * 1000, 1),
                  'Decode (ms/token)': round(r['itl_ms'], 2),
                  'KV cache (MB)': round(r['kv_bytes'] / 1e6, 1)})
    print(f"  {r['in_tokens']:>5} tok -> prefill {r['prefill_sec']*1000:8.1f} ms | "
          f"decode {r['itl_ms']:6.2f} ms/token | KV {r['kv_bytes']/1e6:7.1f} MB")
    del ids
    if HAS_CUDA:
        torch.cuda.empty_cache()

sweep_table = pd.DataFrame(sweep)
sweep_table

In [ ]:
xs = list(sweep_table['Input tokens'])
ax1, ax2 = panels(2)

curves(ax1, [('prefill', xs, list(sweep_table['Prefill (ms)']), COOL)],
       'Prefill grows with the prompt', 'Input tokens', 'ms')
curves(ax2, [('per decoded token', xs, list(sweep_table['Decode (ms/token)']), WARM)],
       'Decode stays flat', 'Input tokens', 'ms per token')
ax2.set_ylim(0, max(sweep_table['Decode (ms/token)']) * 1.6)   # honest zoom: flat looks flat
plt.tight_layout()
plt.show()

print(f"{xs[0]} -> {xs[-1]} input tokens:")
print(f"  prefill   x{sweep_table['Prefill (ms)'].iloc[-1]/sweep_table['Prefill (ms)'].iloc[0]:.1f}")
print(f"  decode/tok x{sweep_table['Decode (ms/token)'].iloc[-1]/sweep_table['Decode (ms/token)'].iloc[0]:.2f}")
print('\nConfirmed. Now: what is decode spending its flat, expensive time on?')

## §5 · The Answer: It Is Reading the Model, Once Per Token

Hypothesis that explains both observations at once: **to produce one token, decode must
read every weight out of memory.** That cost is identical on every step — hence flat —
and it is large — hence slow.

If that is right, decode has a floor set by memory bandwidth, and no amount of faster
arithmetic can go below it. So measure this machine's bandwidth (rather than quote a
spec sheet, which would be wrong on any other GPU) and compare.

In [ ]:
def measure_bandwidth_gbs(iters=20):
    # Big device-to-device copy: each pass reads nbytes and writes nbytes.
    if not HAS_CUDA:
        return None
    free_now, _ = torch.cuda.mem_get_info()
    n_elem = max(int(min(512e6, free_now * 0.15)) // 2, 1 << 20)
    try:
        src = torch.empty(n_elem, dtype=torch.float16, device='cuda')
        dst = torch.empty_like(src)
    except torch.cuda.OutOfMemoryError:
        return None

    for _ in range(3):
        dst.copy_(src)
    best = min(timed(lambda: dst.copy_(src))[1] for _ in range(iters))

    moved = 2 * src.numel() * src.element_size()
    del src, dst
    torch.cuda.empty_cache()
    return moved / best / 1e9


BW_GBS = measure_bandwidth_gbs()
print(f'Measured bandwidth: {BW_GBS:.0f} GB/s (copy benchmark — a practical floor,'
      f' below vendor peak)' if BW_GBS else 'Skipped: needs CUDA.')

In [ ]:
if BW_GBS:
    floor_table = pd.DataFrame([{
        'Prompt': k,
        'Weights read (GB)': round(WEIGHT_BYTES / 1e9, 2),
        'KV read (GB)': round(r['kv_bytes'] / 1e9, 3),
        'Floor (ms/token)': round((WEIGHT_BYTES + r['kv_bytes']) / (BW_GBS * 1e9) * 1000, 2),
        'Measured (ms/token)': round(r['itl_ms'], 2),
        'Bandwidth efficiency':
            f"{(WEIGHT_BYTES + r['kv_bytes'])/(BW_GBS*1e9)*1000/r['itl_ms']*100:.0f}%",
    } for k, r in runs.items()])
    display(floor_table)

    ax, = panels(1, width=9)
    grouped(ax, list(floor_table['Prompt']),
            [('Floor: read weights + KV once', list(floor_table['Floor (ms/token)'])),
             ('Measured per token', list(floor_table['Measured (ms/token)']))],
            f'Decode is bandwidth-bound ({BW_GBS:.0f} GB/s measured)', 'ms per token')
    plt.tight_layout()
    plt.show()

    print(f'Reading {WEIGHT_BYTES/1e9:.1f} GB at {BW_GBS:.0f} GB/s costs '
          f'{WEIGHT_BYTES/(BW_GBS*1e9)*1000:.1f} ms — once per token.')
    print('The gap to measured is launch overhead, attention, sampling, Python: all')
    print('fixable. The floor is not. It is physics plus your model size.')
else:
    floor_table = None
    print('Skipped: needs a CUDA device.')

**Where we are:** decode's cost is one full weight read per token. §4 showed the cache
is what keeps that cost *flat*. So the cache is load-bearing — and it is the thing every
user needs their own copy of. Time to weigh it.

## §6 · Weigh the Cache, and Settle the §2 Prediction

§2 predicted KV bytes per token from `config.json` and promised to check it. The cached
runs from §4 carry real cache objects, so we can weigh them directly — and the sweep
measured the cache at five context lengths.

If prediction and measurement agree, you can size a deployment from `config.json`
without renting anything.

In [ ]:
kv_table = pd.DataFrame([{
    'Prompt': label,
    'Input tokens': r['in_tokens'],
    'Cached tokens': r['kv_tokens'],
    'KV cache (MB)': round(r['kv_bytes'] / 1e6, 2),
    'Measured B/token': round(r['kv_bytes'] / max(r['kv_tokens'], 1)),
    'Predicted B/token': KV_BYTES_PER_TOKEN_PREDICTED,
    'Error': f"{abs(r['kv_bytes']/max(r['kv_tokens'],1) - KV_BYTES_PER_TOKEN_PREDICTED)/KV_BYTES_PER_TOKEN_PREDICTED*100:.1f}%",
} for label, r in runs.items()])
kv_table

In [ ]:
ax1, ax2 = panels(2)

grouped(ax1, [f'{p} prompt' for p in kv_table['Prompt']],
        [('Predicted from config (§2)', [v / 1024 for v in kv_table['Predicted B/token']]),
         ('Measured from the real cache', [v / 1024 for v in kv_table['Measured B/token']])],
        'Was the §2 prediction right?', 'KB per cached token', fmt='{:.0f}')

curves(ax2, [('measured KV cache', list(sweep_table['Input tokens']),
              list(sweep_table['KV cache (MB)']), COOL)],
       'One user\'s cache grows linearly with context', 'Context (tokens)', 'MB')
plt.tight_layout()
plt.show()

err = abs(kv_table['Measured B/token'][0] - KV_BYTES_PER_TOKEN_PREDICTED) / KV_BYTES_PER_TOKEN_PREDICTED
print(f'Prediction error: {err*100:.1f}%')
print('Near zero: the formula is trustworthy, and capacity planning is arithmetic.')
print('Far off: suspect a non-standard cache — sliding window, MLA, quantized KV —')
print('and go read the model code instead of trusting the formula.')
if HAS_CUDA:
    per_user_gb = kv_table['Measured B/token'][0] * LONG_TOKENS / 1e9
    print(f'\nMeasured: {per_user_gb:.2f} GB per user at {LONG_TOKENS} tokens of context.')
    print(f'Free VRAM after weights ({FREE_B/1e9:.1f} GB) therefore seats about '
          f'{int(FREE_B/ (per_user_gb*1e9)) if per_user_gb else 0} of them.')

## §7 · The Chain, Closed

| Step | What we measured | Consequence |
|------|------------------|-------------|
| §1 | Weights = N GB on the device | The fixed cost of being open for business |
| §2 | Weights fixed; KV cache per-user-per-token | Capacity is a KV-cache question, not a weights question |
| §3 | Long prompt waits far longer for token 1 | TTFT is governed by input length |
| §4 | Cache is irrelevant to prefill, decisive for decode | It is a *decode* optimization, not a prompt one |
| §4 | Uncached decode climbs per step; cached decode is flat | The cache turns O(n) per token into O(1) |
| §4 | Even cached, decode is far slower per token than prefill | Two phases, two bottlenecks, never one average |
| §5 | Decode ≈ reading every weight, once per token | A hard floor from bandwidth, not from compute |
| §6 | Real cache size matches the §2 formula | You can size this from `config.json` |

### Two numbers to leave with

1. **prefill tok/s ÷ decode tok/s** on your hardware — the asymmetry that makes batching
   decode nearly free, and makes one giant prompt able to stall a server.
2. **weight bytes ÷ measured bandwidth** — your per-token floor.

### Which is exactly where Demo B starts

The floor is `(weights + KV) / bandwidth`. You cannot change the denominator without
buying a different GPU. So attack the numerator, from both ends:

- **shrink the KV cache** → fewer KV heads (GQA/MQA) or a compressed latent (MLA)
- **shrink the weights** → INT8, INT4

Demo B measures both.

> **Honest limits of this notebook:** one GPU, one model, unbatched pure PyTorch. A real
> engine (vLLM, SGLang, TensorRT-LLM) beats every absolute number here. The *shape* of
> every curve is what transfers — and it is what the rest of the workshop builds on.